In [1]:
# Imports

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)

print("Libraries loaded ✓")

Libraries loaded ✓


# Arte Discreta
### How a Star Is Predicted

*A predictive model of Michelin recognition*

&nbsp;

*"Tell me what you eat, and I will tell you what you are."*  
— Jean Anthelme Brillat-Savarin, *The Physiology of Taste*, 1825

&nbsp;

Avgustina Daskalova · Machine Learning with Python · SoftUni · 2026



## Abstract

For a hundred years the Michelin Guide has awarded stars without 
explaining itself. No published criteria, no named inspectors, no 
reasoning offered. Brillat-Savarin promised that what we eat reveals 
what we are. This project tests the inverse: that what diners write 
reveals what a restaurant will become.

Working only from independent reviews — the language of people who 
never know the verdict in advance — it trains a classifier to separate 
Michelin-recognized restaurants from the rest. Michelin's own prose is 
deliberately excluded; predicting the guide from the guide would prove 
nothing. The reviews are scored against a luxury aspect taxonomy built 
for this project, so the model does not merely guess but reasons in the 
vocabulary of hospitality itself: service, room, precision, the texture 
of an evening.

Because recognition is rare, the problem is one of imbalance, and the 
mathematics follows from that — precision and recall over raw accuracy, 
calibrated probabilities, and an interpretation layer that asks not only 
whether the model is right but why. This is the first of two movements. 
The classical methods here lay the ground for a deeper, multilingual 
version to come.

## Table of Contents

1. **Introduction** — the discreet art, and why it leaves traces
2. **Background** — a century of stars, and prior attempts to predict them
3. **The Mathematics of Judgment** — classification when recognition is rare
4. **The Data** — independent reviews, and what they leave out
5. **A Taxonomy of Luxury** — teaching the model the vocabulary of distinction
6. **Feature Engineering** — from language to signal
7. **Modelling** — four ways of learning to recognize excellence
8. **Interpretation** — what the model saw, and in whose words
9. **Results and Discussion**
10. **Conclusions** — what carries into the second movement
11. **References**

## 1. Introduction

The Michelin Guide has awarded stars since 1926. In a century, the 
criteria have never been published, the inspectors never named, the 
verdict never explained. The opacity is intentional. It is the 
institution's signature, a kind of judgment performed in silence.

Yet judgment leaves traces. Before any restaurant received a star it 
already existed in the world: it served meals, kept hours, and was 
written about by the people who ate there. Diners are not inspectors. 
But they sit at the same tables, and if something extraordinary was 
present, it tends to surface in how they describe the evening, even 
when they cannot name what they noticed.

This project asks whether that ordinary language can be read by a 
machine. From independent reviews alone, it learns to separate the 
restaurants Michelin recognized from those it passed over. The guide's 
own descriptions are kept out entirely: a model that predicts Michelin 
from Michelin learns nothing, only repeats. The interesting question is 
whether strangers, writing for other strangers, leave behind enough to 
anticipate a verdict they never saw.

To give the model more than raw word counts, the reviews are read 
against a taxonomy of luxury written for this project, drawn from the 
working vocabulary of fine hospitality. The model does not only ask 
which words appear; it asks which dimensions of an experience the words 
belong to.

Formally, for each restaurant we predict a label

$$y \in \{0, 1\},$$

where $y = 1$ marks Michelin recognition and $y = 0$ its absence. Because 
recognition is rare, the two classes are deeply unequal, and that 
imbalance shapes every decision that follows — which metrics to trust, 
how to weight the rare class, and how to read a probability honestly.

A note on scope. The data is in English and the methods are 
deliberately classical: no neural networks, no learned embeddings. This 
is the first of two movements. A later version will add depth and other 
languages. For now the aim is exact and modest — to see how far 
ordinary words can carry us toward an extraordinary distinction.

## 2. Background

The first stars were awarded in 1926 — single stars, to a handful of French
tables deemed worth the stop. The three-tier hierarchy we know today arrived in
the early 1930s, the first three-star verdicts handed down in 1933, and in the
near-century since, the grammar of the Michelin Guide has scarcely moved a
comma. What has accumulated is an enormous public ledger of the chosen — which
tables, which rooms, which cities — set against an almost monastic silence about
the reasons. The criteria were not even printed until 1936, and the inspectors
have stayed nameless throughout. Abundance on one side, secrecy on the other. It
is, if you tilt your head, the exact silhouette of a question a machine can be
taught to answer.

The question has been asked before, though usually in a thinner key. The common
approach reaches for what is easy to count: the price bracket, the cuisine, the
postcode, the average rating a restaurant has already gathered like sediment.
Such features predict fame tolerably well — but fame is not what we are hunting.
An adored corner trattoria and a hushed two-star dining room can wear the same
four-and-a-half stars from the crowd. What divides them is not the volume of
praise but its *timbre* — what is noticed, what is named, the words people reach
for when something has quietly moved them.

This project takes that difference as its whole wager. It declines the
structured shortcuts wherever it can afford to, and asks instead whether the
grain of ordinary language — reviews written by people who never knew the
verdict was coming — already carries the secret. Whether, long before the
inspector arrives, diners are describing a starred restaurant in an idiom subtly
their own.

## 3. The Mathematics of Judgment

Stripped of its romance, the task is a *binary classification* problem. Each
restaurant is a point; the label is one bit — recognized, or not. The model
learns a function from the language of reviews to that single bit, and we ask
how often, and how honestly, it is right.

But the arithmetic is lopsided from the start. In our data, the recognized are
a vanishing minority — a few stars among tens of thousands of tables. A lazy
model could call *everything* unrecognized and be correct better than 99.9% of
the time. Accuracy, here, is a flatterer. It rewards the model for ignoring the
very thing we built it to find.

So we judge by other instruments. Writing $TP$ for true positives (starred
restaurants correctly found), $FP$ for false positives (the unstarred, wrongly
anointed), $FN$ for false negatives (the stars the model let slip), and $TN$ for
true negatives (the unstarred, correctly passed over):

$$\text{Precision} = \frac{TP}{TP + FP}, \qquad \text{Recall} = \frac{TP}{TP + FN}$$

**Precision** asks: of the restaurants the model anoints, how many truly hold a
star? **Recall** asks: of the starred restaurants that exist, how many did the
model find? The two pull against each other — cast a wide net and recall rises
while precision falls; be cautious and the reverse. The **F1 score**, their
harmonic mean, holds that tension in a single number:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

The harmonic mean is deliberate: it punishes imbalance between the two, so a
model cannot buy a high score by excelling at one and abandoning the other.

Above these sits the **ROC–AUC**. Where precision and recall judge a single
decision threshold, this measures something larger: the model's ability to
*rank* a starred restaurant above an unstarred one, across every threshold at
once. It is built from two rates —

$$\text{TPR} = \frac{TP}{TP + FN}, \qquad \text{FPR} = \frac{FP}{FP + TN}$$

the **true-positive rate** (recall, again — the share of real stars caught) and
the **false-positive rate** (the share of unstarred restaurants wrongly raised).
As we sweep the decision threshold from strict to lenient, each setting yields a
$(\text{FPR}, \text{TPR})$ pair; plotted together they trace the *ROC curve*.
The **AUC** is the area beneath it:

$$\text{AUC} = \int_0^1 \text{TPR}\; d(\text{FPR})$$

read simply as the probability that the model scores a randomly chosen starred
restaurant higher than a randomly chosen unstarred one. A value of $0.5$ is a
coin toss; $1.0$ is a flawless ranking. Because it never fixes a single
threshold, it survives the imbalance that makes accuracy meaningless.

Two further commitments follow from the rarity of stars. First, the minority
class must be given weight it would never earn by counting alone — through class
weighting or resampling — so the model cannot dismiss it as noise. Second, the
split between training and test data must preserve the proportion of stars in
each, lest the test set, by chance, hold no stars at all and the whole exercise
collapse into measuring nothing. *Stratification* is not a nicety here; it is
structural.

This is the first of two movements. The methods that follow are classical —
logistic regression, tree ensembles, the support vector machine — chosen
because they are legible: when they decide, we can ask them why. The deeper,
multilingual, transformer-driven version waits in the wings, but it would be
premature to reach for depth before the shallow models have spoken.

## 4. The Data

The data comes from two sources. The review text and restaurant 
metadata are drawn from the Yelp Open Dataset, used here under its 
academic licence. The labels — which restaurants hold a Michelin 
star — are public facts, assembled from the Michelin Guide's own 
announcements. This section loads each source, examines its shape, 
and prepares a clean working set.

In [2]:
# Load the business data (one JSON object per line)
business = pd.read_json(RAW / "yelp_academic_dataset_business.json", lines=True)

print(f"Total businesses: {len(business):,}")
business.head()

NameError: name 'RAW' is not defined

In [ ]:
# Which cities does the data cover? Top 20 by business count
business["city"].value_counts().head(20)

In [ ]:
# How many businesses are restaurants?
is_restaurant = business["categories"].str.contains("Restaurant", case=False, na=False)
print(f"Restaurants: {is_restaurant.sum():,} out of {len(business):,}")

In [ ]:
# Keep only restaurants
restaurants = business[is_restaurant].copy()

# Normalize city names: fix "St." vs "Saint" and stray whitespace
restaurants["city"] = (
    restaurants["city"]
    .str.strip()
    .str.replace(r"^St\.?\s+", "Saint ", regex=True)
)

print(f"Restaurants after filtering: {len(restaurants):,}")
restaurants["city"].value_counts().head(15)

In [ ]:
# Save the cleaned restaurant subset to interim
INTERIM.mkdir(parents=True, exist_ok=True)
restaurants.to_parquet(INTERIM / "restaurants.parquet", index=False)

print(f"Saved {len(restaurants):,} restaurants to {INTERIM / 'restaurants.parquet'}")

In [ ]:
# When are the reviews from? Check the date range in the review data.
# We read only the 'date' column, in chunks, so we don't load all 5 GB.
dates = pd.read_json(
    RAW / "yelp_academic_dataset_review.json",
    lines=True,
    chunksize=100_000,
)

min_date, max_date = None, None
for chunk in dates:
    cmin, cmax = chunk["date"].min(), chunk["date"].max()
    min_date = cmin if min_date is None else min(min_date, cmin)
    max_date = cmax if max_date is None else max(max_date, cmax)

print(f"Reviews span from {min_date} to {max_date}")

In [ ]:
# Reload the cleaned restaurants
restaurants = pd.read_parquet(INTERIM / "restaurants.parquet")

# Philadelphia's 2025 Michelin one-star restaurants (public facts)
phl_starred = ["Friday Saturday Sunday", "Her Place Supper Club", "Provenance"]

# Look for them in the Philadelphia Yelp data
phl = restaurants[restaurants["city"] == "Philadelphia"]
matches = phl[phl["name"].str.contains("|".join(phl_starred), case=False, na=False)]

print(f"Philadelphia restaurants in data: {len(phl):,}")
matches[["name", "city", "stars", "review_count"]]

In [ ]:
# Authoritative Michelin star list for cities in our data.
# Source: official Michelin Guide ceremonies (cited in References).
# Facts only — name, city, stars, year first awarded.
michelin = pd.DataFrame([
    {"name": "Friday Saturday Sunday", "city": "Philadelphia", "stars": 1, "year": 2025},
    {"name": "Her Place Supper Club",  "city": "Philadelphia", "stars": 1, "year": 2025},
    {"name": "Provenance",             "city": "Philadelphia", "stars": 1, "year": 2025},
    {"name": "Rocca",                  "city": "Tampa",        "stars": 1, "year": 2023},
    {"name": "Kosen",                  "city": "Tampa",        "stars": 1, "year": 2024},
    {"name": "Koya",                   "city": "Tampa",        "stars": 1, "year": 2024},
    {"name": "Ebbe",                   "city": "Tampa",        "stars": 1, "year": 2024},
    {"name": "Lilac",                  "city": "Tampa",        "stars": 1, "year": 2023},
])

print(f"Michelin-starred restaurants in our cities: {len(michelin)}")
michelin

In [ ]:
# Labeling with fuzzy matching

from rapidfuzz import process, fuzz

restaurants_raw = pd.read_parquet('data/interim/restaurants.parquet')

def normalize(s):
    return s.lower().strip()

michelin['name_norm'] = michelin['name'].apply(normalize)
michelin['city_norm'] = michelin['city'].apply(normalize)
restaurants_raw['name_norm'] = restaurants_raw['name'].apply(normalize)
restaurants_raw['city_norm'] = restaurants_raw['city'].apply(normalize)

matched_indices = set()

for _, mrow in michelin.iterrows():
    city_subset = restaurants_raw[restaurants_raw['city_norm'] == mrow['city_norm']]
    if city_subset.empty:
        continue
    match = process.extractOne(
        mrow['name_norm'],
        city_subset['name_norm'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=75
    )
    if match:
        matched_name, score, idx = match
        real_idx = city_subset.index[city_subset['name_norm'] == matched_name][0]
        matched_indices.add(real_idx)
        print(f"✓ '{mrow['name']}' → '{restaurants_raw.loc[real_idx, 'name']}' (score: {score})")

restaurants_raw['is_michelin'] = 0
restaurants_raw.loc[list(matched_indices), 'is_michelin'] = 1

print(f"\nMichelin positives after fuzzy match: {restaurants_raw['is_michelin'].sum()}")
restaurants_raw.drop(columns=['name_norm', 'city_norm'], inplace=True)
restaurants_raw.to_parquet('data/interim/restaurants_labeled.parquet', index=False)
print("Saved → data/interim/restaurants_labeled.parquet")

In [ ]:
# EDA: Dataset Portrait

df = pd.read_parquet('data/interim/restaurants_labeled.parquet')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Arte Discreta — Dataset Portrait', fontsize=14, style='italic')

# 1. Top cities
city_counts = df['city'].value_counts().head(10)
axes[0].barh(city_counts.index[::-1], city_counts.values[::-1], color='#FF6B9D', edgecolor='white')
axes[0].set_title('Top 10 Cities')
axes[0].set_xlabel('Restaurant count')

# 2. Star rating distribution
df['stars'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='#4ABCE8', edgecolor='white'
)
axes[1].set_title('Yelp Star Distribution')
axes[1].set_xlabel('Stars')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

# 3. Review count distribution
axes[2].hist(df['review_count'].clip(upper=500), bins=50,
             color='#F5A623', edgecolor='white')
axes[2].set_title('Review Count Distribution')
axes[2].set_xlabel('Reviews (clipped at 500)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('figures/eda_dataset_portrait.png', dpi=150, bbox_inches='tight')
plt.show()

print(df[['stars', 'review_count', 'is_open']].describe())
print(f"\nMichelin positives: {df['is_michelin'].sum()} / {len(df)}")


In [ ]:
# Cell 6b — EDA: Michelin vs the Field

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Arte Discreta — Michelin vs the Field', fontsize=14, style='italic')

michelin_pos = df[df['is_michelin'] == 1]
michelin_neg = df[df['is_michelin'] == 0]

# 1. Avg Yelp stars: open vs closed
open_stars = df.groupby('is_open')['stars'].mean()
axes[0].bar(['Closed', 'Open'], open_stars.values,
            color=['#E8674A', '#4ABCE8'], edgecolor='white', width=0.5)
axes[0].set_title('Avg Yelp Stars: Open vs Closed')
axes[0].set_ylabel('Mean Stars')
axes[0].set_ylim(3.0, 4.5)
for i, v in enumerate(open_stars.values):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')

# 2. Review count: Michelin vs non-Michelin
bp = axes[1].boxplot(
    [michelin_neg['review_count'].clip(upper=1000),
     michelin_pos['review_count']],
    labels=['Non-Michelin', 'Michelin'],
    patch_artist=True,
    boxprops=dict(facecolor='#F5A623', alpha=0.8),
    medianprops=dict(color='#1A1A2E', linewidth=2),
    whiskerprops=dict(color='#555555'),
    capprops=dict(color='#555555')
)
axes[1].set_title('Review Count: Michelin vs Field')
axes[1].set_ylabel('Reviews')

# 3. Class imbalance
counts = df['is_michelin'].value_counts().sort_index()
axes[2].bar(['Non-Michelin', 'Michelin'], counts.values,
            color=['#7B68EE', '#FF6B9D'], edgecolor='white', width=0.5)
axes[2].set_title('Class Imbalance')
axes[2].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[2].text(i, v + 200, str(v), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/eda_michelin_vs_field.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
!pip install rapidfuzz --quiet

In [ ]:
raw_dir = 'data/raw'
print("Files in data/raw:")
for f in os.listdir(raw_dir):
    path = os.path.join(raw_dir, f)
    size_mb = os.path.getsize(path) / (1024**2) if os.path.isfile(path) else 0
    print(f"  {f}  —  {size_mb:.1f} MB")